# Sampling Cheat Sheet

Fast reference for probability & non-probability sampling in `numpy` /
`pandas`. Companion to the Extended Lab and Template notebooks.


## Decision guide

| Question | Answer points to |
|---|---|
| Is every unit reachable and roughly equal cost to sample? | **Simple random sampling** |
| Is the population in a (non-periodic) list and you need speed? | **Systematic sampling** |
| Do you know meaningful subgroups (strata) that differ on the outcome? | **Stratified sampling** |
| Are units naturally grouped (stores, classrooms, zip codes) and travel/cost is the bottleneck? | **Cluster sampling** |
| Just need directional, exploratory signal, fast and cheap? | **Convenience sampling** (report it as such — no inference) |
| Need fixed group sizes fast, without full randomization? | **Quota sampling** (report it as such — no inference) |


## Setup boilerplate

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2020)   # swap 2020 for your own seed
# df = pd.read_csv("your_data.csv")  # your population / sampling frame


## Simple random sampling

Formula: sampling fraction `f = n / N`. Every unit: `P(select) = n / N`.


In [2]:
def simple_random_sample(df, n, rng):
    idx = rng.choice(df.index, size=n, replace=False)
    return df.loc[idx]

# sample = simple_random_sample(df, n=100, rng=rng)


## Systematic sampling

Formula: interval `k = N // n`; random start `r ~ Uniform{0, ..., k-1}`;
selected positions = `r, r+k, r+2k, ...`.

**Pitfall:** biased if the list order has a period that lines up with `k`.


In [3]:
def systematic_sample(df, n, rng):
    N = len(df)
    k = N // n
    start = rng.integers(0, k)
    return df.iloc[np.arange(start, N, k)[:n]]

# sample = systematic_sample(df, n=100, rng=rng)


## Stratified sampling

Proportional allocation: `n_h = f * N_h` for each stratum `h`. Draw an
independent SRS of size `n_h` within each stratum.


In [4]:
def stratified_sample(df, strata_col, frac, rng):
    parts = []
    for level, group in df.groupby(strata_col):
        n_g = int(round(frac * len(group)))
        idx = rng.choice(group.index, size=n_g, replace=False)
        parts.append(df.loc[idx])
    return pd.concat(parts)

# sample = stratified_sample(df, strata_col="segment", frac=0.1, rng=rng)


_Need a fixed count per stratum instead of a fraction? Swap `n_g = int(round(frac * len(group)))` for a lookup in a `{level: n_g}` dict._

## Cluster sampling

Randomly select `n_clusters` whole clusters; optionally sub-sample within
each with a second-stage stratified draw (two-stage cluster sampling).


In [5]:
def cluster_sample(df, cluster_col, n_clusters, rng, stage2_frac=None):
    clusters = df[cluster_col].unique()
    chosen = rng.choice(clusters, size=n_clusters, replace=False)
    sample = df[df[cluster_col].isin(chosen)]
    if stage2_frac is not None:
        sample = stratified_sample(sample, cluster_col, stage2_frac, rng)
    return sample

# sample = cluster_sample(df, cluster_col="store_id", n_clusters=8, rng=rng)


## Non-probability: convenience & quota

Use only for exploratory work — **never** for formal inference (no valid
standard errors, confidence intervals, or p-values).


In [6]:
def convenience_sample(df, n, filter_col=None, filter_value=None):
    pool = df if filter_col is None else df[df[filter_col] == filter_value]
    return pool.head(n)

def quota_sample(df, strata_col, quotas):
    parts = [df[df[strata_col] == lvl].head(n_g) for lvl, n_g in quotas.items()]
    return pd.concat(parts)


## Confidence interval for a mean (single sample)

`x_bar +/- t_crit * s / sqrt(n)`, with `t_crit = scipy.stats.t.ppf(1 - alpha/2, df=n-1)`.


In [7]:
from scipy import stats

def mean_ci(sample_values, alpha=0.05):
    x_bar = sample_values.mean()
    s = sample_values.std(ddof=1)
    n_obs = len(sample_values)
    t_crit = stats.t.ppf(1 - alpha / 2, df=n_obs - 1)
    margin = t_crit * s / np.sqrt(n_obs)
    return x_bar - margin, x_bar + margin

# ci_low, ci_high = mean_ci(sample["yield_kg"])


## Quick reference table

| Method | numpy/pandas core call | Random? | Formal inference OK? |
|---|---|---|---|
| Simple random | `rng.choice(index, size=n, replace=False)` | Yes | Yes |
| Systematic | `arange(start, N, k)` | Partially (random start only) | Yes, with caution |
| Stratified | `groupby(strata).apply(SRS)` | Yes, within strata | Yes |
| Cluster | `rng.choice(clusters, size=m, replace=False)` | Yes, at cluster level | Yes (cluster-aware variance) |
| Convenience | `.head(n)` / filter to accessible subset | No | No |
| Quota | fixed count per stratum, non-random pick | No | No |

## Common pitfalls

- Forgetting `replace=False` → the same unit can be drawn twice.
- Rounding stratum sizes (`int(round(...))`) can make totals drift a little
  from your target `n` — check `len(sample)` after stratified/cluster draws.
- Reporting a confidence interval from a convenience or quota sample — the
  math will run, but the interval has no guaranteed coverage.
- Using the population standard deviation (`ddof=0`) instead of the sample
  standard deviation (`ddof=1`) inside a confidence interval formula.
- Cluster sampling variance formulas differ from SRS — a naive SRS-style
  standard error will usually **understate** the true uncertainty.
